# P0 검색·자동분류 검증 실습

국회도서관 API나 OpenAI를 호출하지 않는 **오프라인 검증**입니다. 아래 XML과 관리번호는 테스트용 가상 자료입니다. 실서비스 검색 성공을 뜻하지 않습니다.

이 노트북을 `app.py`와 같은 폴더에서 실행하세요. 각 실습 뒤의 **별도 검증 셀**을 순서대로 실행합니다. 실제 API 점검은 `.env`를 설정한 뒤 별도 `check_live.py`를 사용합니다.

## 1. 준비 — 프로젝트 모듈 불러오기

In [ ]:
from pathlib import Path
import pandas as pd
from xml.sax.saxutils import escape

assert Path("paperlab.py").exists(), "노트북을 app.py가 있는 폴더에서 실행하세요."
from paperlab import parse_search_xml, clean_paper_title
from nanet_metadata import normalize_frame, classify_record, CATEGORY_ORDER
from nanet_search_flow import partition_catalog_records, merge_catalog_pages, search_literature_basic, compare_web_reference
from citation_styles import suggested_citation, format_reference
print("오프라인 실습 준비 완료: API 인증키 불필요")

In [ ]:
# 별도 검증 셀
assert len(CATEGORY_ORDER) == 4
assert CATEGORY_ORDER[0] == "학위논문"
print("검증 통과")

## 2. XML 제목 누락·HTML 태그 정리

In [ ]:
# 예시 필드명이 기사명/저자명이어도 표준 제목으로 변환합니다.
source_rows = [
    {"controlno":"KINX2026000001", "기사명/저자명":"<font color='red'>연습용</font> 기사 / 김연구", "발행자":"<b>연습 학회</b>", "DB":"국내학술기사"},
    {"controlno":"KDMT2026000002", "자료명":"연습용 학위논문", "발행자":"연습대학교 대학원"},
    {"controlno":"MONO2026000003", "자료명":"연습용 도서"},
]
parts = ["<response><header><resultCode>00</resultCode></header><total>3</total>"]
for row in source_rows:
    parts.append("<record>")
    for name, value in row.items():
        parts.append(f"<item><name>{escape(name)}</name><value>{escape(value)}</value></item>")
    parts.append("</record>")
parts.append("</response>")
api_total, records = parse_search_xml("".join(parts).encode("utf-8"))
df = normalize_frame(pd.DataFrame(records))
display(df)

In [ ]:
# 별도 검증 셀
assert api_total == 3
assert df.iloc[0]["자료명/저자사항"] == "연습용 기사 / 김연구"
assert df.iloc[0]["발행자"] == "연습 학회"
assert "<font" not in df.to_string()
print("검증 통과")

## 3. 자동분류·유형별 정렬·충돌 처리

In [ ]:
papers, pending, books, catalog = partition_catalog_records(df)
counts = {kind: int(catalog["자료분류"].eq(kind).sum()) for kind in CATEGORY_ORDER}
conflict = classify_record({"controlno":"KDMT2026000002", "DB":"일반도서"})
print("자료유형별 수집 건수:", counts)
print("상충하는 분류:", conflict)
display(catalog[["자료명/저자사항", "DB", "자료분류", "분류출처"]])

In [ ]:
# 별도 검증 셀
assert catalog["자료분류"].tolist() == ["학위논문", "학술기사", "도서자료"]
assert len(papers) == 2 and len(books) == 1 and pending.empty
assert sum(counts.values()) == len(catalog)
assert conflict.category == "자료유형 미확인" and conflict.status == "충돌"
assert df.iloc[1]["DB"] == "미분류"  # 원본 DB값을 가짜 공식 분류로 덮어쓰지 않습니다.
print("검증 통과")

## 4. 추가 페이지와 중복 제거

In [ ]:
extra = pd.DataFrame([
    {"controlno":"KDMT2026000002", "자료명":"연습용 학위논문"},
    {"controlno":"KINX2026000004", "자료명":"새 연습용 기사", "DB":"국내학술기사"},
])
_, _, _, merged = merge_catalog_pages(catalog, extra)
print("중복 제거 후 자료 수:", len(merged))

In [ ]:
# 별도 검증 셀
assert len(merged) == 4
assert merged["controlno"].is_unique
assert merged.iloc[0]["자료분류"] == "학위논문"
print("검증 통과")

## 5. 검색어가 자동으로 바뀌지 않는지 확인

In [ ]:
# 실제 HTTP 요청이 아니라 가짜 API 함수를 주입합니다.
requested_keywords = []
def fake_api(key, keyword, **options):
    requested_keywords.append(keyword)
    return 1, pd.DataFrame([{"controlno":"MONO2026000003", "DB":"일반도서", "자료명":"연습용 도서"}])
outcome = search_literature_basic("offline-placeholder", "데이터베이스 포렌식", pages=1, fetch=fake_api, automatic_retry=True)
print("호출한 검색어:", requested_keywords)
print("남은 자료:", len(outcome.catalog))

In [ ]:
# 별도 검증 셀
assert requested_keywords == ["데이터베이스 포렌식"]
assert outcome.used_keyword == "데이터베이스 포렌식"
assert len(outcome.books) == 1
assert len(outcome.catalog) == 1  # 학술자료가 없다는 이유로 다른 검색어 결과를 섞지 않습니다.
print("검증 통과")

## 6. 정식 참고문헌 형식 확인

In [ ]:
# 대화에서 지정한 서식 검증 예시입니다. API 실수집 결과가 아닙니다.
example = {
    "자료명/저자사항":"Amazon RDS 데이터베이스 디지털포렌식 조사 기법 연구 / 김수용",
    "DB":"석사학위논문", "발행자":"고려대학교 대학원", "발행연월":"2026년 8월",
}
reference = format_reference(suggested_citation(example))
print(reference)

In [ ]:
# 별도 검증 셀
expected = '김수용, “Amazon RDS 데이터베이스 디지털포렌식 조사 기법 연구”, 석사학위논문, 고려대학교 대학원, 2026년 8월.'
assert reference == expected
assert '연구”,' in reference
print("검증 통과")

## 7. 공식 웹 표본과 수집 목록 비교 원리

In [ ]:
# 아래 역시 가상 표본입니다. 실제 웹 검증은 관리번호를 직접 확인한 경우에만 합니다.
comparison, metrics = compare_web_reference(catalog, [
    {"controlno":"KDMT2026000002", "title":"연습용 학위논문"},
    {"controlno":"KDMT2026000999", "title":"아직 수집하지 않은 예시"},
])
display(comparison)
print(metrics)

In [ ]:
# 별도 검증 셀
assert metrics["비교 기준 관리번호 수"] == 2
assert metrics["현재 수집 목록에서 확인"] == 1
assert metrics["표본 포함률"] == 0.5
assert "API 전체 부재 아님" in comparison.iloc[1]["현재 수집 목록"]
print("검증 통과")

## 실서비스 검증은 별도입니다

`python check_live.py --pages 1`로 실제 승인 키를 이용한 API 점검을 수행할 수 있습니다. 키는 `.env`의 `DATA_GO_KR_API_KEY`에서만 읽고 출력하지 않습니다.

이번 노트북의 성공은 XML·분류·정렬·상태 처리 검증입니다. 공식 웹사이트와 전체 목록 일치, 실제 인증, 상세페이지 접근, Streamlit 브라우저 동작까지 검증한 것은 아닙니다.